# 06 — Low-confidence mining and class-adaptive threshold calibration

**Project:** RAOD++ · **Phase:** F6 · **Protocol:** v1.1

This notebook does **not** generate crops yet. It answers the three questions that must
be settled before any crop is written to disk:

1. What does the confidence distribution of the E0 baseline look like, per class?
2. Where should the mining threshold τ sit, and is a class-adaptive τ justified by data?
3. Do low-confidence regions actually contain real objects, or just background noise?

Question 3 is the load-bearing one. If low-confidence regions are mostly noise, the whole
premise of RAOD++ collapses and we need to know that now, not after building the pipeline.

**Runtime:** ~25–40 min (inference over the training split for 10 seeds). GPU required.


## [Cell 3] Config

Two thresholds are involved and they must not be confused:

- **τ (mining threshold)** — the *upper* bound of the low-confidence band. Detections
  below it are treated as uncertain regions worth enriching. This is what we calibrate here.
- **σ (pseudo-label acceptance)** — a *high* threshold used later when retrieved crops
  enter training. Not touched in this notebook.

`FLOOR` excludes the noise tail: below it, YOLO outputs are dominated by background.

In [1]:
# ===== Cell 3 =====
from pathlib import Path
import numpy as np, pandas as pd, yaml

CONFIG = {
    "PROJECT_ROOT": Path.home() / "RAODpp",
    "EXP": "E0",
    "DATASET": "visdrone",
    "SEEDS": [42, 123, 2024, 7, 3407, 11, 55, 101, 777, 12345],
    "IMGSZ": 640,
    "DEVICE": 0,
    "PREDICT_CONF": 0.01,      # inference floor; band analysis happens afterwards
    "FLOOR": 0.05,             # below this, detections are treated as background noise
    "PERCENTILES": [15, 25, 35],   # protocol v1.0 sweep; 25 is the default
    "DEFAULT_PCT": 25,
    "IOU_MATCH": 0.5,          # for the "is there a real object there?" analysis
    "ALPHA": 0.20,             # context margin for crops (used in notebook 07)
}

ROOT = CONFIG["PROJECT_ROOT"]
OUT = ROOT/"output"; FIGDATA = OUT/"figdata"; FIGDATA.mkdir(parents=True, exist_ok=True)
RUNS = ROOT/"model"/"runs"
DATA_YAML = ROOT/"dataset"/"visdrone_raodpp.yaml"

dcfg = yaml.safe_load(open(DATA_YAML))
BASE = Path(dcfg["path"]); NAMES = dcfg["names"]
TRAIN_IMG = BASE/dcfg["train"]
TRAIN_LBL = Path(str(TRAIN_IMG).replace("/images","/labels"))

print("train images:", len(list(TRAIN_IMG.glob('*.jpg'))))
print("classes     :", len(NAMES))
print("seeds       :", CONFIG["SEEDS"])


train images: 6471
classes     : 10
seeds       : [42, 123, 2024, 7, 3407, 11, 55, 101, 777, 12345]


## [Cell 6] A note on where the mining model has been

Predictions here are made by each E0 model **on its own training split**. Confidences are
therefore inflated relative to unseen data — the model has memorised much of this content.

We use this deliberately rather than working around it. A region the detector is *still*
unsure about after training on it is, by construction, a harder case than one it merely
finds unfamiliar. The manuscript states this explicitly instead of leaving it for a
reviewer to discover. A seed-crossed variant (mine with seed A, train seed B) is kept as a
robustness check for the ablation phase.

## [Cell 6] Inference over the training split
One pass per seed, results cached to `figdata/train_predictions_<run>.csv`. Re-running
skips seeds that are already done.

In [2]:
# ===== Cell 6 (revised: chunked, memory-safe) =====
from ultralytics import YOLO
import time, gc, torch

CHUNK = 250          # images per chunk
MAX_DET = 300        # per-image detection cap

def predict_train(seed):
    run_id = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    out_csv = FIGDATA/f"train_predictions_{run_id}.csv"
    if out_csv.exists():
        print(f"  {run_id}: cached ({sum(1 for _ in open(out_csv))-1:,} rows)")
        return out_csv

    w = RUNS/run_id/"weights"/"best.pt"
    assert w.exists(), f"missing {w}"
    model = YOLO(str(w))
    imgs = sorted(TRAIN_IMG.glob("*.jpg"))
    tmp = out_csv.with_suffix(".part")
    if tmp.exists():
        tmp.unlink()

    t0, total, header = time.time(), 0, True
    for i in range(0, len(imgs), CHUNK):
        chunk = [str(p) for p in imgs[i:i+CHUNK]]
        rows = []
        for r in model.predict(source=chunk, imgsz=CONFIG["IMGSZ"],
                               conf=CONFIG["PREDICT_CONF"], max_det=MAX_DET,
                               device=CONFIG["DEVICE"], stream=True, verbose=False):
            b = r.boxes
            if b is None or len(b) == 0:
                continue
            name = Path(r.path).name
            xyxy = b.xyxy.cpu().numpy(); cls = b.cls.cpu().numpy(); cf = b.conf.cpu().numpy()
            for k in range(len(cf)):
                rows.append((name, int(cls[k]), float(cf[k]),
                             float(xyxy[k][0]), float(xyxy[k][1]),
                             float(xyxy[k][2]), float(xyxy[k][3])))
        if rows:
            pd.DataFrame(rows, columns=["image","class_id","conf","x1","y1","x2","y2"]) \
              .to_csv(tmp, mode="a", header=header, index=False)
            header = False
            total += len(rows)
        del rows
        gc.collect(); torch.cuda.empty_cache()
        print(f"    {min(i+CHUNK, len(imgs))}/{len(imgs)}  rows={total:,}", end="\r")

    tmp.rename(out_csv)
    del model
    gc.collect(); torch.cuda.empty_cache()
    print(f"\n  {run_id}: {total:,} detections in {(time.time()-t0)/60:.1f} min")
    return out_csv

paths = [predict_train(s) for s in CONFIG["SEEDS"]]
print("\ndone:", len(paths), "seeds")

    6471/6471  rows=1,186,566
  E0_visdrone_s42: 1,186,566 detections in 1.7 min
    6471/6471  rows=1,193,289
  E0_visdrone_s123: 1,193,289 detections in 1.6 min
    6471/6471  rows=1,186,269
  E0_visdrone_s2024: 1,186,269 detections in 1.6 min
    6471/6471  rows=1,187,950
  E0_visdrone_s7: 1,187,950 detections in 1.6 min
    6471/6471  rows=1,175,192
  E0_visdrone_s3407: 1,175,192 detections in 1.6 min
    6471/6471  rows=1,183,350
  E0_visdrone_s11: 1,183,350 detections in 1.6 min
    6471/6471  rows=1,183,303
  E0_visdrone_s55: 1,183,303 detections in 1.6 min
    6471/6471  rows=1,195,296
  E0_visdrone_s101: 1,195,296 detections in 1.6 min
    6471/6471  rows=1,190,773
  E0_visdrone_s777: 1,190,773 detections in 1.6 min
    6471/6471  rows=1,175,739
  E0_visdrone_s12345: 1,175,739 detections in 1.6 min

done: 10 seeds


## [Cell 8] Confidence distribution per class
Establishes whether one global τ can serve every class. If the per-class distributions sit
at visibly different levels, a single threshold systematically over-mines some classes and
under-mines others — which is exactly the argument the adaptive-threshold literature makes
(CascadeMatch's APM, FreeMatch's class-local thresholds).

In [3]:
# ===== Cell 8 =====
ref_seed = CONFIG["SEEDS"][0]
ref_id = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{ref_seed}"
det = pd.read_csv(FIGDATA/f"train_predictions_{ref_id}.csv")
det = det[det.conf >= CONFIG["FLOOR"]]
det["class_name"] = det.class_id.map(NAMES)

stats = (det.groupby(["class_id","class_name"]).conf
         .agg(n="size", mean="mean", median="median",
              p15=lambda s: s.quantile(.15), p25=lambda s: s.quantile(.25),
              p35=lambda s: s.quantile(.35), p75=lambda s: s.quantile(.75))
         .round(4).reset_index())
stats.to_csv(FIGDATA/"conf_distribution_per_class.csv", index=False)
print(stats.to_string(index=False))
print(f"\nglobal p25 = {det.conf.quantile(.25):.4f}   (single fixed threshold would use this)")
print(f"spread of per-class p25: {stats.p25.min():.3f} - {stats.p25.max():.3f}")


 class_id      class_name      n   mean  median    p15    p25    p35    p75
        0      pedestrian 158771 0.2202  0.1483 0.0675 0.0834 0.1041 0.2979
        1          people  29733 0.1798  0.1333 0.0669 0.0813 0.0990 0.2346
        2         bicycle  14003 0.1594  0.1084 0.0610 0.0707 0.0826 0.1970
        3             car 293090 0.3584  0.2496 0.0760 0.1037 0.1451 0.6156
        4             van  24868 0.4199  0.3721 0.0936 0.1370 0.2026 0.7024
        5           truck  21419 0.3237  0.1942 0.0698 0.0890 0.1167 0.5427
        6        tricycle   9118 0.2057  0.1322 0.0652 0.0782 0.0959 0.2670
        7 awning-tricycle   4822 0.2392  0.1606 0.0702 0.0872 0.1083 0.3362
        8             bus   9499 0.3996  0.2944 0.0774 0.1075 0.1564 0.7207
        9           motor  76190 0.1962  0.1324 0.0653 0.0788 0.0962 0.2545

global p25 = 0.0904   (single fixed threshold would use this)
spread of per-class p25: 0.071 - 0.137


## [Cell 10] Class-adaptive τ, averaged over seeds
τ for class c is the p-th percentile of that class's confidence distribution, averaged
across the 10 baseline models so the threshold does not depend on one seed's quirks.
Computed for all three percentiles in the protocol sweep.

In [4]:
# ===== Cell 10 =====
rows = []
for seed in CONFIG["SEEDS"]:
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    d = pd.read_csv(FIGDATA/f"train_predictions_{rid}.csv")
    d = d[d.conf >= CONFIG["FLOOR"]]
    for c, g in d.groupby("class_id"):
        for p in CONFIG["PERCENTILES"]:
            rows.append({"seed": seed, "class_id": int(c), "pct": p,
                         "tau": float(g.conf.quantile(p/100))})

tau_long = pd.DataFrame(rows)
tau = (tau_long.groupby(["class_id","pct"]).tau.agg(["mean","std"]).reset_index()
       .rename(columns={"mean":"tau","std":"tau_std"}))
tau["class_name"] = tau.class_id.map(NAMES)
tau.to_csv(FIGDATA/"tau_class_adaptive.csv", index=False)

wide = tau.pivot(index="class_name", columns="pct", values="tau").round(4)
print("class-adaptive tau by percentile (mean over 10 seeds)")
print(wide.to_string())
print(f"\nseed-to-seed std of tau: max = {tau.tau_std.max():.4f}  (stability check)")


class-adaptive tau by percentile (mean over 10 seeds)
pct                  15      25      35
class_name                             
awning-tricycle  0.0693  0.0865  0.1092
bicycle          0.0609  0.0705  0.0829
bus              0.0792  0.1094  0.1568
car              0.0761  0.1039  0.1457
motor            0.0659  0.0798  0.0977
pedestrian       0.0677  0.0839  0.1049
people           0.0679  0.0829  0.1012
tricycle         0.0662  0.0803  0.0987
truck            0.0695  0.0896  0.1192
van              0.0922  0.1376  0.2070

seed-to-seed std of tau: max = 0.0046  (stability check)


## [Cell 12] Does the low-confidence band contain real objects?

The premise test. For the reference seed we take every detection inside the band
[FLOOR, τ_class] and ask whether it overlaps a ground-truth box at IoU ≥ 0.5, and whether
that ground-truth object was *already* covered by a confident detection. The interesting
quantity is the third row: real objects that only the low-confidence band caught.

In [5]:
# ===== Cell 12 =====
def iou_matrix(a, b):
    """a: (N,4), b: (M,4) in xyxy -> (N,M) IoU"""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:,None,0], b[None,:,0]); y1 = np.maximum(a[:,None,1], b[None,:,1])
    x2 = np.minimum(a[:,None,2], b[None,:,2]); y2 = np.minimum(a[:,None,3], b[None,:,3])
    inter = np.clip(x2-x1, 0, None) * np.clip(y2-y1, 0, None)
    ar_a = (a[:,2]-a[:,0]) * (a[:,3]-a[:,1]); ar_b = (b[:,2]-b[:,0]) * (b[:,3]-b[:,1])
    return inter / (ar_a[:,None] + ar_b[None,:] - inter + 1e-9)

from PIL import Image
tau_map = dict(zip(tau[tau.pct==CONFIG["DEFAULT_PCT"]].class_id,
                   tau[tau.pct==CONFIG["DEFAULT_PCT"]].tau))

det_all = pd.read_csv(FIGDATA/f"train_predictions_{ref_id}.csv")
sample_images = sorted(det_all.image.unique())[:1500]      # subsample for speed
n_band = n_band_hit = n_band_only = n_conf = 0

for img in sample_images:
    d = det_all[det_all.image == img]
    lp = TRAIN_LBL/(Path(img).stem + ".txt")
    if not lp.exists():
        continue
    W, H = Image.open(TRAIN_IMG/img).size
    gt = []
    for line in open(lp):
        f = line.split()
        if len(f) != 5: continue
        c = int(f[0]); cx, cy, bw, bh = map(float, f[1:])
        gt.append([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H, c])
    if not gt: continue
    gt = np.array(gt)

    d = d.assign(tau=d.class_id.map(tau_map).fillna(0.3))
    band = d[(d.conf >= CONFIG["FLOOR"]) & (d.conf < d.tau)]
    conf = d[d.conf >= d.tau]
    n_band += len(band); n_conf += len(conf)

    if len(band):
        m = iou_matrix(band[["x1","y1","x2","y2"]].values, gt[:, :4])
        hit = m.max(axis=1) >= CONFIG["IOU_MATCH"]
        n_band_hit += int(hit.sum())
        if len(conf):
            mc = iou_matrix(conf[["x1","y1","x2","y2"]].values, gt[:, :4])
            covered = (mc >= CONFIG["IOU_MATCH"]).any(axis=0)     # GT already detected
            best_gt = m.argmax(axis=1)
            n_band_only += int(((m.max(axis=1) >= CONFIG["IOU_MATCH"]) &
                                (~covered[best_gt])).sum())
        else:
            n_band_only += int(hit.sum())

res = pd.DataFrame([
    ["images analysed", len(sample_images)],
    ["confident detections (conf >= tau)", n_conf],
    ["low-confidence band detections", n_band],
    ["band detections matching a GT object (IoU>=0.5)", n_band_hit],
    ["band precision (share hitting real objects)", round(n_band_hit/max(1,n_band), 4)],
    ["GT objects recovered ONLY by the band", n_band_only],
], columns=["quantity","value"])
res.to_csv(FIGDATA/"lowconf_band_analysis.csv", index=False)
print(res.to_string(index=False))


                                       quantity       value
                                images analysed   1500.0000
             confident detections (conf >= tau) 143035.0000
                 low-confidence band detections  42362.0000
band detections matching a GT object (IoU>=0.5)   9097.0000
    band precision (share hitting real objects)      0.2147
          GT objects recovered ONLY by the band   2936.0000


## [Cell 14] Crop budget: what each design option would cost

Three ways to build the crop set, with different rigour/cost trade-offs. The numbers below
are measured, not guessed, so the choice can be made on evidence.

In [6]:
# ===== Cell 14 =====
per_seed = []
for seed in CONFIG["SEEDS"]:
    rid = f"{CONFIG['EXP']}_{CONFIG['DATASET']}_s{seed}"
    d = pd.read_csv(FIGDATA/f"train_predictions_{rid}.csv")
    d = d.assign(tau=d.class_id.map(tau_map).fillna(0.3))
    n = int(((d.conf >= CONFIG["FLOOR"]) & (d.conf < d.tau)).sum())
    per_seed.append({"seed": seed, "band_crops": n})

ps = pd.DataFrame(per_seed)
mean_crops = ps.band_crops.mean()
KB = 15  # ~15 KB per 224x224 JPEG crop

opts = pd.DataFrame([
    ["A: seed-matched (each seed mines its own)", mean_crops*len(CONFIG["SEEDS"]),
     "most rigorous; 10x CLIP + storage"],
    ["B: single mining model (seed 42)", mean_crops,
     "cheapest; enrichment tied to one seed"],
    ["C: consensus across seeds", mean_crops,
     "one crop set, robust; needs cross-model box matching"],
], columns=["option","crops","note"])
opts["storage_GB"] = (opts.crops*KB/1e6).round(2)
opts["clip_minutes"] = (opts.crops/8000).round(1)   # ~8k crops/min on RTX 4500 (fp16)

print(ps.to_string(index=False))
print(f"\nmean band crops per seed: {mean_crops:,.0f}")
print()
print(opts.to_string(index=False))
opts.to_csv(FIGDATA/"crop_budget_options.csv", index=False)


 seed  band_crops
   42      162000
  123      162913
 2024      160344
    7      162384
 3407      158846
   11      161761
   55      160193
  101      163214
  777      162779
12345      159667

mean band crops per seed: 161,410

                                   option     crops                                                 note  storage_GB  clip_minutes
A: seed-matched (each seed mines its own) 1614101.0                    most rigorous; 10x CLIP + storage       24.21         201.8
         B: single mining model (seed 42)  161410.1                cheapest; enrichment tied to one seed        2.42          20.2
                C: consensus across seeds  161410.1 one crop set, robust; needs cross-model box matching        2.42          20.2


**Decision point.** Report back: the per-class τ table, the band analysis (especially
*band precision* and *GT objects recovered only by the band*), and the crop budget. The
crop-set option (A / B / C) is chosen from these numbers, then notebook 07 writes the
crops and builds the CLIP + FAISS index.

In [7]:
# ===== Cell 15 — precision & recall as a function of confidence =====
from PIL import Image

GRID = [0.05,0.075,0.10,0.15,0.20,0.25,0.30,0.40,0.50,0.60,0.70,0.80]
N_IMG = 1500

det_all = pd.read_csv(FIGDATA/f"train_predictions_{ref_id}.csv")
imgs = sorted(det_all.image.unique())[:N_IMG]

n_gt_tot = 0
matched_at = {t: 0 for t in GRID}      # GT matched by detections with conf >= t
ndet_at    = {t: 0 for t in GRID}      # detections with conf >= t
nhit_at    = {t: 0 for t in GRID}      # of those, how many match a GT

for img in imgs:
    lp = TRAIN_LBL/(Path(img).stem + ".txt")
    if not lp.exists(): continue
    W, H = Image.open(TRAIN_IMG/img).size
    gt = []
    for line in open(lp):
        f = line.split()
        if len(f) != 5: continue
        cx, cy, bw, bh = map(float, f[1:])
        gt.append([(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H])
    if not gt: continue
    gt = np.array(gt); n_gt_tot += len(gt)

    d = det_all[det_all.image == img]
    if not len(d): continue
    M = iou_matrix(d[["x1","y1","x2","y2"]].values, gt)
    conf = d.conf.values
    for t in GRID:
        sel = conf >= t
        ndet_at[t] += int(sel.sum())
        if sel.any():
            sub = M[sel]
            nhit_at[t] += int((sub.max(axis=1) >= 0.5).sum())
            matched_at[t] += int((sub.max(axis=0) >= 0.5).sum())

curve = pd.DataFrame([{
    "conf_threshold": t,
    "n_detections": ndet_at[t],
    "precision": round(nhit_at[t]/max(1, ndet_at[t]), 4),
    "recall": round(matched_at[t]/max(1, n_gt_tot), 4),
} for t in GRID])
curve["marginal_recall_gain"] = (curve.recall - curve.recall.shift(-1)).round(4)
curve.to_csv(FIGDATA/"precision_recall_vs_confidence.csv", index=False)
print(f"GT objects in {len(imgs)} images: {n_gt_tot:,}\n")
print(curve.to_string(index=False))

GT objects in 1500 images: 105,255

 conf_threshold  n_detections  precision  recall  marginal_recall_gain
          0.050        185397     0.5130  0.6697                0.0171
          0.075        155188     0.5727  0.6526                0.0169
          0.100        134472     0.6211  0.6357                0.0376
          0.150        105988     0.6979  0.5981                0.0382
          0.200         87046     0.7583  0.5599                0.0400
          0.250         72862     0.8105  0.5199                0.0396
          0.300         61934     0.8546  0.4803                0.0791
          0.400         46145     0.9180  0.4012                0.0789
          0.500         34951     0.9575  0.3223                0.0740
          0.600         26345     0.9787  0.2483                0.0686
          0.700         18992     0.9897  0.1797                0.0736
          0.800         11217     0.9938  0.1061                   NaN


In [8]:
# ===== Cell 17 — precision-calibrated, class-adaptive band =====
from PIL import Image

GRID = np.array([0.05,0.075,0.10,0.15,0.20,0.25,0.30,0.40,0.50,0.60,0.70,0.80,0.90])
TARGET_LOW, TARGET_HIGH = 0.50, 0.90
N_IMG = 1500

det_all = pd.read_csv(FIGDATA/f"train_predictions_{ref_id}.csv")
imgs = sorted(det_all.image.unique())[:N_IMG]

# per class, per bin: detections and hits
nd = {c: np.zeros(len(GRID)-1) for c in NAMES}
nh = {c: np.zeros(len(GRID)-1) for c in NAMES}

for img in imgs:
    lp = TRAIN_LBL/(Path(img).stem + ".txt")
    if not lp.exists(): continue
    W, H = Image.open(TRAIN_IMG/img).size
    gt, gtc = [], []
    for line in open(lp):
        f = line.split()
        if len(f) != 5: continue
        c = int(f[0]); cx, cy, bw, bh = map(float, f[1:])
        gt.append([(cx-bw/2)*W,(cy-bh/2)*H,(cx+bw/2)*W,(cy+bh/2)*H]); gtc.append(c)
    if not gt: continue
    gt = np.array(gt); gtc = np.array(gtc)

    d = det_all[det_all.image == img]
    if not len(d): continue
    M = iou_matrix(d[["x1","y1","x2","y2"]].values, gt)
    for c in d.class_id.unique():
        m = (d.class_id == c).values
        same = (gtc == c)
        best = M[m][:, same].max(axis=1) if same.any() else np.zeros(m.sum())
        confs = d.conf.values[m]
        idx = np.digitize(confs, GRID) - 1
        for b in range(len(GRID)-1):
            sel = idx == b
            if sel.any():
                nd[c][b] += sel.sum()
                nh[c][b] += (best[sel] >= 0.5).sum()

rows = []
centers = (GRID[:-1] + GRID[1:]) / 2
for c in NAMES:
    p = nh[c] / np.maximum(nd[c], 1)
    valid = nd[c] >= 30                      # ignore bins with too little evidence
    lo = hi = np.nan
    if valid.any():
        pv, cv = p[valid], centers[valid]
        lo = float(np.interp(TARGET_LOW, pv, cv)) if pv.max() >= TARGET_LOW else float(cv[-1])
        hi = float(np.interp(TARGET_HIGH, pv, cv)) if pv.max() >= TARGET_HIGH else float(cv[-1])
    rows.append({"class_id": c, "class_name": NAMES[c],
                 "tau_low": round(lo,3), "tau_high": round(hi,3),
                 "n_det": int(nd[c].sum())})

band = pd.DataFrame(rows)
band.to_csv(FIGDATA/"tau_precision_calibrated.csv", index=False)
print(band.to_string(index=False))

# crop budget under the new band
tot = 0
for c in NAMES:
    d = det_all[det_all.class_id == c]
    r = band[band.class_id == c].iloc[0]
    if np.isnan(r.tau_low): continue
    tot += int(((d.conf >= r.tau_low) & (d.conf < r.tau_high)).sum())
print(f"\ncrops in calibrated band (per seed, full train): {tot*6471/N_IMG:,.0f}")

 class_id      class_name  tau_low  tau_high  n_det
        0      pedestrian    0.309     0.611  46253
        1          people    0.244     0.750  13488
        2         bicycle    0.322     0.650   5205
        3             car    0.338     0.703  74919
        4             van    0.304     0.722   3542
        5           truck    0.283     0.711   5203
        6        tricycle    0.325     0.727   5451
        7 awning-tricycle    0.464     0.799   2434
        8             bus    0.307     0.704    815
        9           motor    0.329     0.716  27389

crops in calibrated band (per seed, full train): 561,389
